# Attention, Learn to Solve Routing Problems! — Walkthrough

Kool, van Hoof, Welling (ICLR 2019) · [arXiv:1803.08475](https://arxiv.org/abs/1803.08475)

This notebook maps each part of the paper to the code in `src/`, with runnable
sanity checks. **Tiny dimensions** are used so everything runs on CPU in seconds —
the architecture is identical to the paper, only the numbers shrink.

Run from the project root (`attention_learn_to_route/`).


## 1. Setup

We use the real classes from `src/`. The only change vs. the paper is a smaller
config (`d_h`, `n_layers`, batch) so the notebook is instant on CPU.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

from src.model import MultiHeadAttention, AttentionLayer, Encoder, Decoder, AttentionModel, ModelConfig
from src.data import generate_tsp_instances
from src.utils import tsp_tour_length
from src.loss import reinforce_loss

torch.manual_seed(0)

# Toy config — paper uses d_h=128, n_layers=3, n=20/50/100, batch=512.
cfg = ModelConfig(d_h=32, d_x=2, n_layers=2, n_heads=4, d_ff=64, tanh_clip=10.0)
BATCH, N = 3, 12   # 3 instances of TSP-12
print("toy config:", cfg)

## 2. Multi-Head Attention as message passing (Appendix A, eqs 10–14)

> "We interpret the attention mechanism by Vaswani et al. (2017) as a weighted
> message passing algorithm between nodes in a graph. ... we compute the key
> $k_i$, value $v_i$ and query $q_i$ for each node by projecting the embedding."
> — Appendix A

$$q_i = W^Q h_i,\quad k_i = W^K h_i,\quad v_i = W^V h_i \qquad u_{ij} = \frac{q_i^\top k_j}{\sqrt{d_k}}$$
$$a_{ij} = \operatorname{softmax}_j(u_{ij}),\qquad h_i' = \sum_j a_{ij} v_j$$

with $M = 8$ heads and $d_k = d_v = d_h / M$ (App. A; printed as "$M d_h$" but the
consistent reading is $d_h/M = 16$).


In [ ]:
mha = MultiHeadAttention(cfg.d_h, cfg.n_heads, bias=cfg.qkv_bias)
h = torch.randn(BATCH, N, cfg.d_h)         # (batch, n, d_h)
out = mha(h)
assert out.shape == (BATCH, N, cfg.d_h), out.shape
print(f"✓ MHA: {tuple(h.shape)} -> {tuple(out.shape)}; head dim d_k = {mha.d_k}")
assert mha.d_k == cfg.d_h // cfg.n_heads

## 3. Encoder attention layer: skip-connection + Batch Norm (§3.1)

> "Each sublayer adds a skip-connection and batch normalization (BN) (which we
> found to work better than layer normalization)."
> — §3.1

$$\hat h_i = \mathrm{BN}\big(h_i + \mathrm{MHA}_i(h_1,\dots,h_n)\big),\qquad
  h_i = \mathrm{BN}\big(\hat h_i + \mathrm{FF}(\hat h_i)\big)$$

The feed-forward sublayer has hidden dim 512 with ReLU (App. A). Note this is
**add-then-norm** (post-norm), and **BatchNorm**, not LayerNorm.


In [ ]:
layer = AttentionLayer(cfg)
out = layer(h)
assert out.shape == (BATCH, N, cfg.d_h), out.shape
# Batch norm is used (not layer norm):
assert isinstance(layer.bn1, nn.BatchNorm1d) and isinstance(layer.bn2, nn.BatchNorm1d)
print(f"✓ AttentionLayer: {tuple(h.shape)} -> {tuple(out.shape)} (BatchNorm1d x2, FF dim {cfg.d_ff})")

## 4. Encoder: input projection, N layers, graph embedding (§3.1)

> "From the $d_x$-dimensional input features $x_i$ ... the encoder computes initial
> $d_h$-dimensional node embeddings $h_i^{(0)} = W^x x_i + b^x$ ... updated using
> $N$ attention layers. ... The encoder computes an aggregated embedding
> $\bar h^{(N)}$ of the input graph as the mean of the final node embeddings."
> — §3.1


In [ ]:
enc = Encoder(cfg)
coords = generate_tsp_instances(BATCH, N)      # (batch, n, 2) in [0,1]^2
node_emb, graph_emb = enc(coords)
assert node_emb.shape == (BATCH, N, cfg.d_h)
assert graph_emb.shape == (BATCH, cfg.d_h)
# Graph embedding is exactly the mean of node embeddings.
assert torch.allclose(graph_emb, node_emb.mean(dim=1), atol=1e-6)
print(f"✓ Encoder: coords {tuple(coords.shape)} -> nodes {tuple(node_emb.shape)}, graph {tuple(graph_emb.shape)}")

## 5. Decoder: context node, glimpse, pointer (§3.2)

> "the context of the decoder ... consists of the embedding of the graph, the
> previous (last) node $\pi_{t-1}$ and the first node $\pi_1$. For $t = 1$ we use
> learned $d_h$-dimensional parameters $v^l$ and $v^f$."
> — §3.2

> "we add one final decoder layer with a single attention head ... we clip the
> result (before masking!) within $[-C, C]$ ($C = 10$) using $\tanh$."
> — §3.2

The decoder rolls out the full tour autoregressively. `greedy` takes the argmax
action; `sampling` draws from the policy (used by REINFORCE).


In [ ]:
dec = Decoder(cfg)
tour, log_lik = dec(node_emb, graph_emb, decode_type="greedy")
assert tour.shape == (BATCH, N) and log_lik.shape == (BATCH,)
# Every tour must be a valid permutation of the n nodes.
for i in range(BATCH):
    assert sorted(tour[i].tolist()) == list(range(N)), "tour is not a permutation!"
print(f"✓ Decoder: tour {tuple(tour.shape)} (valid permutations), log_lik {tuple(log_lik.shape)}")
print("  example tour:", tour[0].tolist())

## 6. Full Attention Model + cost $L(\pi)$ (§3, §4)

The cost of a TSP solution is the Euclidean length of the closed tour. Sampling
should, in expectation, cost no less than greedy on an untrained model, but both
are far from optimal until training.


In [ ]:
model = AttentionModel(cfg)
g_tour, _ = model(coords, "greedy")
s_tour, s_ll = model(coords, "sampling")
g_cost = tsp_tour_length(coords, g_tour)
s_cost = tsp_tour_length(coords, s_tour)
print(f"✓ Full model params: {sum(p.numel() for p in model.parameters()):,}")
print("  greedy cost  :", [round(c, 3) for c in g_cost.tolist()])
print("  sample cost  :", [round(c, 3) for c in s_cost.tolist()])

## 7. REINFORCE with baseline (§4, Algorithm 1)

> "$\nabla L(\theta|s) = \mathbb{E}_{p_\theta(\pi|s)}\big[(L(\pi) - b(s))\,
> \nabla \log p_\theta(\pi|s)\big]$"
> — §4

The advantage $L(\pi) - b(s)$ is **detached** (it is a constant weight on the
log-likelihood). Here we use greedy cost as a stand-in baseline $b(s)$.


In [ ]:
loss = reinforce_loss(cost=s_cost, baseline=g_cost, log_likelihood=s_ll)
model.zero_grad()
loss.backward()
n_with_grad = sum(p.grad is not None and p.grad.norm() > 0 for p in model.parameters())
n_total = sum(1 for _ in model.parameters())
print(f"✓ REINFORCE loss = {loss.item():.4f}; params receiving gradient: {n_with_grad}/{n_total}")
assert n_with_grad > 0

## 8. One real training step

A single optimizer step with the paper's pieces: Adam, gradient clipping
(max_norm=1.0, from official code), greedy-rollout-style baseline.


In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
before = tsp_tour_length(coords, model(coords, "greedy")[0]).mean().item()
for _ in range(20):
    c = generate_tsp_instances(64, N)
    tour, ll = model(c, "sampling")
    cost = tsp_tour_length(c, tour)
    baseline = tsp_tour_length(c, model(c, "greedy")[0])
    loss = reinforce_loss(cost, baseline, ll)
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # [FROM_OFFICIAL_CODE]
    opt.step()
after = tsp_tour_length(coords, model(coords, "greedy")[0]).mean().item()
print(f"✓ greedy cost on held-out batch: {before:.3f} -> {after:.3f} after 20 steps")

## 9. Common pitfalls

1. **BatchNorm, not LayerNorm** (§3.1). The paper explicitly found BN works better.
   BN normalizes over the batch×nodes dimension — `eval()` mode matters for the
   deterministic greedy rollout baseline.

2. **`d_k = d_h / M`, not `M·d_h`** (App. A). The printed "$M d_h = 16$" is a
   typesetting artifact; with $d_h=128, M=8$ only $128/8 = 16$ is consistent.

3. **Clip *before* masking** (§3.2). The single-head pointer logits are
   $C\tanh(\cdot)$ and only *then* are visited nodes set to $-\infty$. Clipping
   after masking would turn $-\infty$ into $\pm C$.

4. **No positional encoding** (§3.1). Node embeddings must be order-invariant; do
   not add positional encodings to the encoder.

5. **Detach the advantage** (§4). $L(\pi) - b(s)$ is a constant weight; if you let
   gradients flow through the baseline or the cost you get the wrong estimator.

6. **Baseline update is conditional** (§4, Alg. 1). Replace $\theta^{BL}$ only when
   a paired t-test ($\alpha = 5\%$) says the current policy is significantly better.

7. **First-epoch warmup** (§5). Use an exponential ($\beta=0.8$) baseline in epoch 1,
   then switch to the greedy rollout baseline.
